In [1]:
import torch
import pandas as pd
import json
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"

print("Loading Base Model for Zero-Shot Evaluation...")
processor = AutoProcessor.from_pretrained(model_id)
# Load in 4-bit or 8-bit to fit in 12GB VRAM during inference
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    # quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map=device,
)
# Load your unseen test dataset
df_test = pd.read_csv("./data/combined_multimodal_dataset_test.csv")
results = []
BATCH_SIZE = 4
print(f"Running Batched Baseline Inference (Batch Size: {BATCH_SIZE})...")
for i in tqdm(range(0, len(df_test), BATCH_SIZE), desc="Zero-Shot Evaluation"):
    batch_df = df_test.iloc[i:i+BATCH_SIZE]
    conversations = []
    commands = []
    # Build batch
    for _, row in batch_df.iterrows():
        audio_path = f"./data/synthesized_test/{row['Audio_File']}"
        conversations.append([
            {"role": "user", "content": [{"type": "audio", "path": audio_path}]}
        ])
        commands.append(row["User_Command"])

    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True, # If true tells model to generate the Assistant response
    ).to(device, dtype=torch.bfloat16)
    # Generate the response
    outputs = model.generate(
        **inputs,
        do_sample=False, # Disable sampling for faster inference, enable to use temp and top_k
        temperature=0.2,
        top_p=0.95,
        max_new_tokens=256
    )
    decoded_outputs = processor.batch_decode(
        outputs[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    for command, output in zip(commands, decoded_outputs):
        # Attempt to parse JSON (This will likely fail in the baseline)
        valid_json = False
        try:
            # Crude extraction: look for braces
            json_str = output[output.find("{"):output.rfind("}")+1]
            if json_str: # Ensure string is not empty before parsing
                json.loads(json_str)
                valid_json = True
        except:
            pass

        results.append({
            "Command": command,
            "Base_Model_Output": output.strip(),
            "Valid_JSON": valid_json
        })

# Save for thesis comparison
df_results = pd.DataFrame(results)
df_results.to_csv("./models/baseline_results.csv", index=False)
success_rate = df_results["Valid_JSON"].mean() * 100
print(f"\nBaseline Zero-Shot JSON Success Rate: {success_rate:.2f}%")

Loading Base Model for Zero-Shot Evaluation...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Running Batched Baseline Inference (Batch Size: 4)...


Zero-Shot Evaluation:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Baseline Zero-Shot JSON Success Rate: 0.00%
